# Embed Tahoe data with pretrained AE

Load our pretrained AE (trained on Tahoe HVGs) and compute 128-dim latent embeddings
for the tahoe h5ad files. Save as `obsm['AE_128_pretrained']`.

**Key steps:**
1. Load HVG var_names from the zarr the model was trained on
2. Load each tahoe h5ad (all 62710 genes), subset to the 6087 HVGs
3. Run the AE encoder to get 128-dim embeddings
4. Save back to h5ad

In [1]:
import json
import os
import numpy as np
import scanpy as sc
import torch
from tqdm import tqdm

import sys
sys.path.insert(0, '/lustre/groups/ml01/workspace/xiaotong.fu/reconstruction/reconstruction/src')
from sc_reconstruction.models.reconae import Autoencoder

/home/icb/xiaotong.fu/miniconda3/envs/cstm_scvi_env/lib/python3.12/site-packages/scanpy/_utils/__init__.py:35: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/home/icb/xiaotong.fu/miniconda3/envs/cstm_scvi_env/lib/python3.12/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/home/icb/xiaotong.fu/miniconda3/envs/cstm_scvi_env/lib/python3.12/site-packages/scanpy/readwrite.py:15: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


## 1. Config

In [9]:
CHECKPOINT_PATH = '/lustre/groups/ml01/workspace/xiaotong.fu/reconstruction/weights/tahoe/split03/AE/Default/400_[4096, 4096, 4096]_32_20251103/epoch=99-val/loss_epoch=0.0360.ckpt'
ZARR_ATTRS_PATH = '/lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/tahoe/comb_w_obs_3000wCCM.zarr/.zattrs'
DATA_DIR = '/lustre/groups/ml01/workspace/xiaotong.fu/data/pancellflow/unipert'

TAHOE_FILES = {
    'tahoe_a549': 'tahoe_a549_w_emb.h5ad',
    'tahoe_aspc_1': 'tahoe_aspc_1_w_emb.h5ad',
    'tahoe_h4': 'tahoe_h4_w_emb.h5ad',
    'tahoe_hop62': 'tahoe_hop62_w_emb.h5ad',
    'tahoe_panc_1': 'tahoe_panc_1_w_emb.h5ad',
    'tahoe_snu_1': 'tahoe_snu_1_w_emb.h5ad',
    'tahoe_sw48': 'tahoe_sw48_w_emb.h5ad',
}

EMBED_KEY = 'AE_32_pretrained'
BATCH_SIZE = 4096
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Device: cpu


## 2. Load HVG var_names from zarr

In [6]:
with open(ZARR_ATTRS_PATH) as f:
    zarr_attrs = json.load(f)

hvg_var_names = zarr_attrs['var_names']
print(f'HVG var_names: {len(hvg_var_names)} genes')
print(f'First 10: {hvg_var_names[:10]}')

HVG var_names: 6087 genes
First 10: ['GCLC', 'CFTR', 'KLHL13', 'MTMR7', 'SLC7A2', 'CD38', 'HSPB6', 'PDK4', 'ABCB5', 'CALCR']


## 3. Load pretrained AE

In [4]:
model = Autoencoder.load_from_checkpoint(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)
model.eval()
model = model.to(DEVICE)

# Verify input dim matches HVG count
first_weight = next(model.encoder.encoder.parameters())
input_dim = first_weight.shape[1]
print(f'Model input dim: {input_dim}')
print(f'HVG count: {len(hvg_var_names)}')
assert input_dim == len(hvg_var_names), f'Mismatch: model expects {input_dim}, zarr has {len(hvg_var_names)} genes'

Model input dim: 6087
HVG count: 6087


## 4. Embed each tahoe file

In [5]:
@torch.no_grad()
def encode_adata(model, adata, hvg_var_names, batch_size=4096, device='cpu'):
    """Subset adata to HVGs and encode in batches."""
    # Find matching genes
    adata_var = set(adata.var_names)
    common = [g for g in hvg_var_names if g in adata_var]
    missing = [g for g in hvg_var_names if g not in adata_var]
    print(f'  Gene matching: {len(common)}/{len(hvg_var_names)} found, {len(missing)} missing')
    if missing:
        print(f'  Missing genes (first 10): {missing[:10]}')
    
    # Subset to HVGs in the correct order
    # For missing genes, fill with zeros
    n_cells = adata.n_obs
    n_genes = len(hvg_var_names)
    
    # Build index mapping: hvg_var_names[i] -> adata.var_names index
    var_name_to_idx = {name: idx for idx, name in enumerate(adata.var_names)}
    gene_indices = [var_name_to_idx.get(g, -1) for g in hvg_var_names]
    
    embeddings = []
    for start in tqdm(range(0, n_cells, batch_size), desc='  Encoding'):
        end = min(start + batch_size, n_cells)
        
        # Get chunk and convert to dense if sparse
        chunk = adata.X[start:end]
        if hasattr(chunk, 'toarray'):
            chunk = chunk.toarray()
        chunk = np.asarray(chunk, dtype=np.float32)
        
        # Reorder to HVG order, zero-fill missing
        batch_hvg = np.zeros((chunk.shape[0], n_genes), dtype=np.float32)
        for i, idx in enumerate(gene_indices):
            if idx >= 0:
                batch_hvg[:, i] = chunk[:, idx]
        
        x = torch.from_numpy(batch_hvg).to(device)
        z = model.encode(x)
        embeddings.append(z.cpu().numpy())
    
    return np.concatenate(embeddings, axis=0)

In [ ]:
for name, fname in TAHOE_FILES.items():
    path = os.path.join(DATA_DIR, fname)
    print(f'\n{"=" * 60}')
    print(f'Processing: {name} ({fname})')
    
    adata = sc.read_h5ad(path)
    print(f'  Loaded: {adata.n_obs:,} cells x {adata.n_vars} genes')
    
    emb = encode_adata(model, adata, hvg_var_names, batch_size=BATCH_SIZE, device=DEVICE)
    print(f'  Embedding shape: {emb.shape}')
    
    adata.obsm[EMBED_KEY] = emb
    adata.uns['AE_pretrained_hvg_var_names'] = hvg_var_names
    adata.write_h5ad(path)
    print(f'  Saved {EMBED_KEY} + hvg_var_names to {fname}')
    del adata, emb

print(f'\n{"=" * 60}')
print('ALL DONE')


Processing: tahoe_a549 (tahoe_a549_w_emb.h5ad)


## 5. Verify

In [4]:
import anndata as ad

test_path = os.path.join(DATA_DIR, list(TAHOE_FILES.values())[0])
adata = ad.read_h5ad(test_path, backed='r')
print(f'File: {os.path.basename(test_path)}')
print(f'obsm keys: {list(adata.obsm.keys())}')
if EMBED_KEY in adata.obsm:
    emb = adata.obsm[EMBED_KEY]
    print(f'{EMBED_KEY}: shape={emb.shape}, dtype={emb.dtype}')
    print(f'  values[:3, :5]:\n{emb[:3, :5]}')
del adata

File: tahoe_a549_w_emb.h5ad
obsm keys: ['AE_10_pretrained', 'AE_128_pretrained', 'AE_32_pretrained', 'X_scconcept', 'X_scgpt', 'X_scimilarity', 'X_scimilarity_correct', 'X_state']
AE_32_pretrained: shape=(2326890, 32), dtype=float32
  values[:3, :5]:
[[  6.7588153  -14.801589     1.6322933    3.3682885   -4.287105  ]
 [  3.2762587   -7.5159283   -0.9140891    2.6170099    0.03466005]
 [  6.75828     -2.7166142   -1.0033329    4.4489737    1.1375409 ]]


## 6. Quick quality check: UMAP colored by cell line

Sample cells from each tahoe file and plot UMAP on the AE embeddings.

In [7]:
# Verify stored hvg_var_names matches the zarr source
test_path = os.path.join(DATA_DIR, list(TAHOE_FILES.values())[0])
adata = ad.read_h5ad(test_path, backed='r')

stored_hvg = adata.uns.get('AE_pretrained_hvg_var_names', None)
if stored_hvg is None:
    print('AE_pretrained_hvg_var_names NOT in uns — re-run cell 10')
else:
    match = list(stored_hvg) == list(hvg_var_names)
    print(f'Stored hvg_var_names length: {len(stored_hvg)}')
    print(f'Zarr hvg_var_names length:   {len(hvg_var_names)}')
    print(f'Order matches: {match}')
    if not match:
        for i, (a, b) in enumerate(zip(stored_hvg, hvg_var_names)):
            if a != b:
                print(f'  First mismatch at index {i}: stored={a}, zarr={b}')
                break

del adata

Stored hvg_var_names length: 6087
Zarr hvg_var_names length:   6087
Order matches: True


In [10]:
import anndata as ad
import pandas as pd
import matplotlib.pyplot as plt

N_SAMPLE = 2000  # cells per cell line

rng = np.random.default_rng(42)
parts = []
for name, fname in TAHOE_FILES.items():
    path = os.path.join(DATA_DIR, fname)
    adata = ad.read_h5ad(path, backed='r')
    emb = np.array(adata.obsm[EMBED_KEY])
    n = min(N_SAMPLE, len(emb))
    idx = rng.choice(len(emb), n, replace=False)
    cl_name = name.replace('tahoe_', '').upper()
    sub = ad.AnnData(
        X=emb[idx],
        obs=pd.DataFrame({'cell_line': cl_name}, index=[f'{cl_name}_{i}' for i in range(n)]),
    )
    parts.append(sub)
    print(f'{cl_name}: sampled {n} cells')
    del adata

adata_umap = ad.concat(parts)
print(f'\nCombined: {adata_umap.n_obs} cells, {adata_umap.obs["cell_line"].nunique()} cell lines')

sc.pp.pca(adata_umap, n_comps=50)
sc.pp.neighbors(adata_umap, n_pcs=50)
sc.tl.umap(adata_umap)

fig, ax = plt.subplots(figsize=(8, 7))
sc.pl.umap(adata_umap, color='cell_line', ax=ax, show=False, title=f'AE_128_pretrained ({N_SAMPLE}/cell line)')
plt.tight_layout()
plt.show()

del parts, adata_umap

A549: sampled 2000 cells
ASPC_1: sampled 2000 cells
H4: sampled 2000 cells
HOP62: sampled 2000 cells
PANC_1: sampled 2000 cells
SNU_1: sampled 2000 cells
SW48: sampled 2000 cells

Combined: 14000 cells, 7 cell lines


/home/icb/xiaotong.fu/miniconda3/envs/cstm_scvi_env/lib/python3.12/site-packages/scanpy/preprocessing/_pca/__init__.py:245: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  Version(ad.__version__) < Version("0.9")


ValueError: n_components=50 must be between 1 and min(n_samples, n_features)=32 with svd_solver='arpack'